# Memory-Augmented Brain Encoding — End-to-End Run
### `Mrabbi3/memory-augmented-brain-encoding`

**This single notebook does everything in one run:** downloads the real Algonauts fMRI + multimodal features, aligns them, and runs the temporal-integration sweep that maps where in cortex longer context helps. It downloads to the **runtime's local disk** (not Google Drive), so there's no dependence on any previous session or Drive account — just run it top to bottom.

**Prereq:** a Colab secret named `HF_TOKEN` (your Hugging Face read token), with Llama-3.2 access granted on HF. To set it: click the 🔑 icon in the left sidebar → add `HF_TOKEN` → toggle access for this notebook.

**Time:** ~10–15 min for one subject (the default), mostly downloading. Let it run.

> Run each cell in order with Shift+Enter, or use **Runtime → Run all**.

## 1. Configuration

In [ ]:
import numpy as np, os, glob

SUBJECTS  = ["sub-01"]                 # add "sub-02","sub-03","sub-05" later (needed for the noise ceiling in NB02)
TASKS     = ["friends"]
SEASONS   = ["s1", "s2", "s3"]         # Friends seasons (set to ["s1"] for an even faster first run)
BACKBONES = ["vjepa2_avg_feat", "whisper", "Llama-3.2-3B"]   # video / audio / text
LAYER_KEYS = {
    "vjepa2_avg_feat": "encoder.layer.25.mlp.fc2_avg",   # -> 1408
    "whisper":         "layers.25.fc2",                  # -> 1280
    "Llama-3.2-3B":    "model.layers.15",                # -> 3072
}
HRF_SHIFT_TRS   = 3
TR_SECONDS      = 1.49
CONTEXT_LENGTHS = [1, 2, 4, 8, 16, 32, 64, 100]   # TRs of causal context; capacity is held fixed across all of them
N_PCA           = 250
RIDGE_ALPHAS    = np.logspace(2, 6, 9)
TEST_FRAC       = 0.2
SEED            = 0

# All data goes to LOCAL runtime disk -> fast, and no Drive account confusion.
DATA_DIR     = "/content/data"
FMRI_DIR     = f"{DATA_DIR}/fmri"
FEATURES_DIR = f"{DATA_DIR}/features"
ALIGNED_DIR  = f"{DATA_DIR}/aligned"
RESULTS_DIR  = "/content/results"
for d in [FMRI_DIR, FEATURES_DIR, ALIGNED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Optional: copy the small results file to Drive at the very end (safe to leave on; failures are ignored).
SAVE_RESULTS_TO_DRIVE = True
DRIVE_RESULTS = "/content/drive/MyDrive/Research/memory-brain-encoding/results"

FMRI_REPO = "https://github.com/courtois-neuromod/algonauts_2025.competitors.git"
FEAT_REPO = "medarc/algonauts_2025.features"
print("Config ready. Subjects:", SUBJECTS, "| Seasons:", SEASONS)

## 2. Install dependencies
A current `git-annex` (Ubuntu's apt version is too old for DataLad), then the Python libraries.

In [ ]:
%%bash
git config --global user.name  "MD Rabbi"
git config --global user.email "mrifat205@gmail.com"
cd /usr/local
wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba >/dev/null 2>&1
export MAMBA_ROOT_PREFIX=/usr/local
./bin/micromamba install -y -p /usr/local -c conda-forge git-annex >/dev/null 2>&1
echo "git-annex ->"; git-annex version | head -1

In [ ]:
!pip install -q datalad huggingface_hub h5py nilearn scikit-learn matplotlib numpy
print("python deps ready")

## 3. Hugging Face login

In [ ]:
from huggingface_hub import login, whoami
from google.colab import userdata
try:
    login(token=userdata.get("HF_TOKEN"))
    print("HF logged in as:", whoami()["name"])
except Exception as e:
    print("Could not read HF_TOKEN. Open the Secrets panel (key icon, left sidebar), add a secret named "
          "HF_TOKEN with your Hugging Face read token, toggle access for THIS notebook, then re-run this cell.")
    raise

## 4. Download fMRI (DataLad → local disk)
CC0-licensed Algonauts/CNeuroMod data. We pull only the small `fmri/` folder and skip the 109 GB of raw stimuli (the features below replace it).

In [ ]:
import os, glob
os.chdir("/content")
if not os.path.isdir("/content/algonauts_2025.competitors"):
    !datalad install {FMRI_REPO}
os.chdir("/content/algonauts_2025.competitors")
for sub in SUBJECTS:
    print(f"--- fMRI {sub} ---")
    !datalad get -r -J4 fmri/{sub}
src = glob.glob("/content/algonauts_2025.competitors/fmri/**/*.h5", recursive=True)
for f in src:
    dest = os.path.join(FMRI_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        !cp -L "{f}" "{dest}"
got = sorted(glob.glob(FMRI_DIR + "/*.h5"))
print(f"\n{len(got)} fMRI file(s):")
for f in got: print("  ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.0f} MB")

## 5. Download features (Hugging Face → local disk)
Real TRIBE-style backbones: V-JEPA 2 (video), Whisper (audio), Llama-3.2-3B (text), one file per episode.

In [ ]:
from huggingface_hub import snapshot_download
patterns = []
for bk in BACKBONES:
    for task in TASKS:
        if task == "friends":
            for s in SEASONS: patterns.append(f"{bk}/friends/{s}/*")
        else: patterns.append(f"{bk}/{task}/*")
snapshot_download(repo_id=FEAT_REPO, repo_type="dataset", allow_patterns=patterns, local_dir=FEATURES_DIR)
nf = len(glob.glob(FEATURES_DIR + "/**/*.h5", recursive=True))
print(f"downloaded {nf} feature files")
for bk in BACKBONES:
    print(f"  {bk}: {len(glob.glob(FEATURES_DIR + f'/{bk}/**/*.h5', recursive=True))} episodes")

## 6. Align features ↔ fMRI
For each episode: take the chosen layer per backbone, flatten `(T,1,D)->(T,D)`, match to the fMRI segment, trim to a common length, apply the HRF lag, z-score, and save a per-subject `.npz` to local disk.

In [ ]:
import h5py, re

def find_fmri(sub, task):
    c = [f for f in glob.glob(FMRI_DIR + "/*.h5") if sub in f and task in os.path.basename(f).lower()]
    return c[0] if c else None
def episode_id(name):
    m = re.search(r"s\d{2}e\d{2}[a-d]?", os.path.basename(str(name)))
    return m.group(0) if m else None
def load_feature(path, key):
    with h5py.File(path, "r") as h:
        if key not in h: return None
        a = np.asarray(h[key][:], dtype=np.float32)
    return a.reshape(a.shape[0], -1)
def fmri_episode_map(p):
    with h5py.File(p, "r") as h:
        return {episode_id(k): k for k in h.keys() if episode_id(k)}
def feature_path(bk, ep, task="friends"):
    c = [p for p in glob.glob(FEATURES_DIR + f"/{bk}/{task}/**/*.h5", recursive=True) if episode_id(p) == ep]
    return c[0] if c else None

def build_subject(sub, task="friends"):
    fp = find_fmri(sub, task)
    if fp is None: print(f"  [skip] no fMRI {sub}"); return
    fmap = fmri_episode_map(fp)
    feat_eps = {episode_id(p) for p in glob.glob(FEATURES_DIR + f"/{BACKBONES[0]}/{task}/**/*.h5", recursive=True)}
    feat_eps.discard(None)
    eps = sorted(feat_eps & set(fmap.keys()))
    Xs, ys, bounds, used = [], [], [0], []
    with h5py.File(fp, "r") as fh:
        for ep in eps:
            feats, ok = [], True
            for bk in BACKBONES:
                p = feature_path(bk, ep, task)
                a = load_feature(p, LAYER_KEYS[bk]) if p else None
                if a is None: ok = False; break
                feats.append(a)
            if not ok: continue
            yseg = np.asarray(fh[fmap[ep]][:], dtype=np.float32)
            T = min([f.shape[0] for f in feats] + [yseg.shape[0]])
            Xe = np.concatenate([f[:T] for f in feats], axis=1)[:T - HRF_SHIFT_TRS]
            ye = yseg[:T][HRF_SHIFT_TRS:T]
            Xs.append(Xe); ys.append(ye); bounds.append(bounds[-1] + Xe.shape[0]); used.append(ep)
    if not Xs: print(f"  [skip] no aligned episodes {sub}"); return
    X = np.concatenate(Xs, 0); y = np.concatenate(ys, 0)
    X = (X - X.mean(0)) / (X.std(0) + 1e-8); y = (y - y.mean(0)) / (y.std(0) + 1e-8)
    out = os.path.join(ALIGNED_DIR, f"{sub}_{task}_aligned.npz")
    np.savez_compressed(out, X=X.astype(np.float32), y=y.astype(np.float32),
                        bounds=np.array(bounds), episodes=np.array(used))
    print(f"  {sub}: {len(used)} eps -> X {X.shape}, y {y.shape}")
    del X, y, Xs, ys

for sub in SUBJECTS:
    build_subject(sub, "friends")
print("\naligned files:", [os.path.basename(f) for f in sorted(glob.glob(ALIGNED_DIR + "/*.npz"))])

## 7. The temporal-integration sweep
Fit one shared PCA on the training episodes, then for each context length `L` pool features over the past `L` TRs (within episodes), fit ridge, and record the per-parcel test correlation. Capacity is identical across all `L`, so differences come only from temporal integration.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
import time

def make_split(episodes, test_frac=TEST_FRAC, seed=SEED):
    uniq = sorted(set(episodes.tolist())); rng = np.random.default_rng(seed)
    perm = rng.permutation(len(uniq)); n = max(1, int(round(len(uniq) * test_frac)))
    return set(uniq[i] for i in perm[:n])
def tr_masks(eps, bounds, test_ids):
    T = int(bounds[-1]); te = np.zeros(T, bool)
    for i, ep in enumerate(eps):
        if ep in test_ids: te[int(bounds[i]):int(bounds[i+1])] = True
    return ~te, te
def causal_pool(Xp, bounds, L):
    if L == 1: return Xp
    out = np.empty_like(Xp)
    for i in range(len(bounds) - 1):
        a, b = int(bounds[i]), int(bounds[i+1]); seg = Xp[a:b]
        cs = np.cumsum(seg, 0); pooled = cs.copy(); pooled[L:] = cs[L:] - cs[:-L]
        counts = np.minimum(np.arange(1, len(seg) + 1), L)[:, None]
        out[a:b] = pooled / counts
    return out
def col_corr(p, t):
    p = p - p.mean(0); t = t - t.mean(0)
    return (p * t).sum(0) / (np.sqrt((p**2).sum(0) * (t**2).sum(0)) + 1e-8)

files = sorted(glob.glob(ALIGNED_DIR + "/*_friends_aligned.npz"))
assert files, "no aligned files — re-run cell 6"
d0 = np.load(files[0]); test_ids = make_split(d0["episodes"])
X0, b0, e0 = d0["X"], d0["bounds"], d0["episodes"]; trm0, _ = tr_masks(e0, b0, test_ids)
scaler = StandardScaler().fit(X0[trm0])
pca = PCA(n_components=N_PCA, svd_solver="randomized", random_state=SEED).fit(scaler.transform(X0[trm0]))
print(f"PCA: {N_PCA} comps explain {pca.explained_variance_ratio_.sum()*100:.1f}% of feature variance")
del X0

all_r = np.zeros((len(files), len(CONTEXT_LENGTHS), 1000), np.float32); subs = []
for si, f in enumerate(files):
    sub = os.path.basename(f).split("_")[0]; subs.append(sub)
    d = np.load(f); X, y, bounds, eps = d["X"], d["y"], d["bounds"], d["episodes"]
    trm, tem = tr_masks(eps, bounds, test_ids); Xp = pca.transform(scaler.transform(X))
    t0 = time.time()
    for li, L in enumerate(CONTEXT_LENGTHS):
        Xpool = causal_pool(Xp, bounds, L)
        m = RidgeCV(alphas=RIDGE_ALPHAS, alpha_per_target=True).fit(Xpool[trm], y[trm])
        all_r[si, li] = col_corr(m.predict(Xpool[tem]), y[tem])
    print(f"{sub}: {time.time()-t0:.0f}s | mean r @L=1 {all_r[si,0].mean():.3f}, best-L mean r {all_r[si].max(0).mean():.3f}")
    del X, y, Xp
print("\nsweep done. all_r:", all_r.shape)

## 8. Result — does longer context help, and where?

In [ ]:
from nilearn import datasets
import matplotlib.pyplot as plt

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else l for l in atlas["labels"]]
NETS = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]
def net_of(l):
    for n in NETS:
        if n in l: return n
    return "Other"
networks = np.array([net_of(l) for l in labels])

r_mean = all_r.mean(0); xs = np.array(CONTEXT_LENGTHS)
plt.figure(figsize=(9, 5.5))
for n in NETS:
    idx = networks == n
    if idx.sum() == 0: continue
    plt.plot(xs, r_mean[:, idx].mean(1), marker="o", label=f"{n} ({idx.sum()})")
plt.xscale("log", base=2); plt.xticks(xs, xs)
plt.xlabel("context length L (TRs, 1 TR = 1.49 s)"); plt.ylabel("mean test correlation r")
plt.title("Encoding accuracy vs. temporal context, by network")
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
opt_L = np.array(CONTEXT_LENGTHS)[r_mean.argmax(0)]
print("Median optimal context length (TRs) per network  [sensory should be short, association longer]:")
for n in sorted(NETS, key=lambda n: np.median(opt_L[networks == n]) if (networks == n).any() else 0):
    idx = networks == n
    if idx.sum() == 0: continue
    print(f"  {n:12s}: median L = {int(np.median(opt_L[idx])):3d}   (best-r {r_mean[:, idx].max(0).mean():.3f})")

## 9. Save results

In [ ]:
out = os.path.join(RESULTS_DIR, "sweep_r.npz")
np.savez_compressed(out, all_r=all_r, context_lengths=np.array(CONTEXT_LENGTHS),
                    subjects=np.array(subs), networks=networks, test_ids=np.array(sorted(test_ids)))
print("saved", out)
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive
        try: drive.mount("/content/drive")
        except ValueError: pass
        os.makedirs(DRIVE_RESULTS, exist_ok=True)
        import shutil; shutil.copy(out, os.path.join(DRIVE_RESULTS, "sweep_r.npz"))
        print("copied results to Drive:", DRIVE_RESULTS)
    except Exception as e:
        print("Drive save skipped (results still in /content/results, downloadable via the Files panel):", e)

## What's next
If the network curves show sensory regions (Vis/SomMot) peaking at short context while association regions (Default/Cont) keep benefiting from longer `L`, you've replicated the cortical temporal hierarchy with modern multimodal features — a clean result. **NB02** will divide these correlations by a per-parcel **noise ceiling** (needs ≥2 subjects, so add the others to `SUBJECTS`) to report fraction of explainable variance, and make the formal cortical map. **NB03** is where your learned memory module competes against this fixed-window baseline.